<a href="https://colab.research.google.com/github/jdeepak-4u/my-new-ai-repo/blob/feature-agent/classify_agent_mcp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Problem Statement - Classification Agent with MCP

## Objective

Build an intent classification agent using:

- MCP (Model Context Protocol)
- LangChain
- Azure OpenAI
- Langfuse

The agent should classify user queries into:

- `info`
- `action`
- `summary`

and route requests to the correct MCP tools.

---

## Hackathon Tasks

You need to:

1. Install required packages
2. Configure Azure OpenAI + Langfuse
3. Create MCP tools
4. Start MCP server
5. Connect MCP client
6. Build an intent classifier
7. Evaluate classifier accuracy
8. Build routing agent
9. Improve prompt quality
10. Observe traces in Langfuse

# Step 1 - Install Packages

In [1]:
!pip install -qU   mcp   langchain   langchain-core   langgraph   langchain-mcp-adapters   langchain-groq   langchain-openai   langfuse   pydantic   pandas   scikit-learn   nest_asyncio   python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.4/236.4 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 482.4/482.4 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.3/472.3 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 36.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not curr

# Step 2 - Import Packages

In [2]:
import os
import getpass

# Step 3 - Configure Environment Variables

Set:

- Groq AI credentials
- Langfuse credentials

In [3]:
def set_secret_if_missing(env_name: str, required: bool = False, default: str | None = None):
    if os.environ.get(env_name):
        print(f"{env_name}: already set")
        return
    if default is not None:
        os.environ[env_name] = default
        print(f"{env_name}: set to default")
        return
    if required:
        os.environ[env_name] = getpass.getpass(f"Enter {env_name}: ")
    else:
        value = getpass.getpass(f"Enter {env_name} or leave blank to skip: ")
        if value:
            os.environ[env_name] = value

# Required for Groq classifier
set_secret_if_missing("GROQ_API_KEY", required=True)

# Optional Langfuse config
set_secret_if_missing("LANGFUSE_PUBLIC_KEY")
set_secret_if_missing("LANGFUSE_SECRET_KEY")
set_secret_if_missing("LANGFUSE_BASE_URL", default="https://us.cloud.langfuse.com")

print("Configuration cell complete.")


Enter GROQ_API_KEY: ··········
Enter LANGFUSE_PUBLIC_KEY or leave blank to skip: ··········
Enter LANGFUSE_SECRET_KEY or leave blank to skip: ··········
LANGFUSE_BASE_URL: set to default
Configuration cell complete.


## Step 4 Initialize Langfuse tracing

If Langfuse keys are configured, LangChain invocations will emit traces through `CallbackHandler`.

In [4]:
from langfuse import get_client
from langfuse.langchain import CallbackHandler

LANGFUSE_ENABLED = bool(os.environ.get("LANGFUSE_PUBLIC_KEY") and os.environ.get("LANGFUSE_SECRET_KEY"))

if LANGFUSE_ENABLED:
    langfuse = get_client()
    langfuse_handler = CallbackHandler()
    print("Langfuse tracing enabled.")
else:
    langfuse = None
    langfuse_handler = None
    print("Langfuse keys not found. Tracing will be skipped, but the notebook will still run.")


def lc_config(run_name: str, tags: list[str] | None = None):
    cfg = {"run_name": run_name, "tags": tags or []}
    if langfuse_handler:
        cfg["callbacks"] = [langfuse_handler]
    return cfg

Langfuse tracing enabled.


# Step 5 - Create MCP Server and register MCP Tools

Create an MCP server using FastMCP.

In [5]:
%%writefile mcp_intent_tools_server.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("IntentRoutingTools")


@mcp.tool()
def info_tool(query: str) -> str:
    """Use this tool for info intent: answer factual, explanatory, or how-to questions."""
    return (
        "INFO TOOL RESULT\n"
        f"Received query: {query}\n"
        "Suggested handling: provide facts, definitions, explanation, or guidance."
    )


@mcp.tool()
def action_tool(query: str) -> str:
    """Use this tool for action intent: perform, create, update, book, send, schedule, or execute something."""
    return (
        "ACTION TOOL RESULT\n"
        f"Received request: {query}\n"
        "Suggested handling: execute or prepare the requested operation after validation."
    )


@mcp.tool()
def summary_tool(text: str) -> str:
    """Use this tool for summary intent: summarize, condense, extract key points, or produce a recap."""
    clean = " ".join(text.split())
    words = clean.split()
    preview = " ".join(words[:35]) + ("..." if len(words) > 35 else "")
    return (
        "SUMMARY TOOL RESULT\n"
        f"Input word count: {len(words)}\n"
        f"Concise preview: {preview}"
    )


if __name__ == "__main__":
    mcp.run(transport="stdio")


Writing mcp_intent_tools_server.py


In [6]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

In [7]:
from contextlib import AsyncExitStack

class MCPClient:
    def __init__(self, server_script):
        self.server_script = server_script
        self.stack = AsyncExitStack()
        self.session = None
        self.tools = []
        self.tool_by_name = {}
        self.stderr_log = "mcp_server_stderr.log"

# Step 6 Connect MCP client and load tools

The MCP stdio client starts `mcp_intent_tools_server.py` as a subprocess, initializes the MCP session, and loads the server tools as LangChain tools.

In [8]:
import asyncio
import nest_asyncio
from contextlib import AsyncExitStack
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools

nest_asyncio.apply()

class MCPToolClient:
    def __init__(self, server_script: str):
        self.server_script = server_script
        self.stack = AsyncExitStack()
        self.session = None
        self.tools = []
        self.tool_by_name = {}

    # async def connect(self):
    #     server_params = StdioServerParameters(
    #         command="python",
    #         args=[self.server_script],
    #     )
    #     read, write = await self.stack.enter_async_context(stdio_client(server_params))
    #     self.session = await self.stack.enter_async_context(ClientSession(read, write))
    #     await self.session.initialize()
    #     self.tools = await load_mcp_tools(self.session)
    #     self.tool_by_name = {tool.name: tool for tool in self.tools}
    #     return self.tools
    async def connect(self):
        server_params = StdioServerParameters(
            command="python",
            args=[self.server_script],
        )

        # In Colab/Jupyter, sys.stderr may not support fileno().
        # MCP stdio_client uses stderr during subprocess startup, so pass a real file.
        self.stderr_log = getattr(self, "stderr_log", "mcp_server_stderr.log")
        self._errlog_handle = open(self.stderr_log, "a", buffering=1)

        # Ensure the log file is closed when the AsyncExitStack is closed.
        self.stack.callback(self._errlog_handle.close)

        read, write = await self.stack.enter_async_context(
            stdio_client(server_params, errlog=self._errlog_handle)
        )

        self.session = await self.stack.enter_async_context(ClientSession(read, write))
        await self.session.initialize()

        self.tools = await load_mcp_tools(self.session)
        self.tool_by_name = {tool.name: tool for tool in self.tools}

        return self.tools

    async def close(self):
        await self.stack.aclose()

mcp_client = MCPToolClient("mcp_intent_tools_server.py")
tools = asyncio.get_event_loop().run_until_complete(mcp_client.connect())
print("Loaded MCP tools:", [tool.name for tool in tools])

Loaded MCP tools: ['info_tool', 'action_tool', 'summary_tool']


# Step 7 - Build Intent Classifier V1

Create a very basic classifier prompt.

The model should return ONLY:

- info
- action
- summary

In [9]:
from typing import Literal
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate

class IntentPrediction(BaseModel):
    intent: Literal["info", "action", "summary"] = Field(description="The intent class.")
    confidence: float = Field(ge=0.0, le=1.0, description="Classifier confidence from 0 to 1.")
    rationale: str = Field(description="One short sentence explaining the classification.")

# Pick a Groq model that supports structured output. Change if your account uses a different model.
GROQ_MODEL = os.environ.get("GROQ_MODEL", "llama-3.3-70b-versatile")

llm = ChatGroq(
    model=GROQ_MODEL,
    temperature=0,
    max_retries=2,
)

structured_llm = llm.with_structured_output(IntentPrediction)

initial_classifier_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You classify a user's request into exactly one label:
- info: the user wants an answer, explanation, facts, comparison, or guidance.
- action: the user wants something performed, created, changed, sent, booked, scheduled, routed, or executed.
- summary: the user wants content summarized, condensed, recapped, or key points extracted.
Return only the structured classification.
"""),
    ("human", "User query: {query}")
])

initial_classifier = initial_classifier_prompt | structured_llm

async def classify_intent(query: str, classifier=initial_classifier, run_name: str = "intent-classifier") -> IntentPrediction:
    return await classifier.ainvoke({"query": query}, config=lc_config(run_name, ["intent-classification"]))

sample_query = "Summarize this customer feedback into three bullet points."
pred = asyncio.get_event_loop().run_until_complete(classify_intent(sample_query))
pred


IntentPrediction(intent='summary', confidence=0.9, rationale='The user wants a summary of customer feedback.')


# Step 8 - Build Routing Agent

The agent should:

1. Classify query
2. Select correct tool
3. Return response

In [10]:
INTENT_TO_TOOL = {
    "info": "info_tool",
    "action": "action_tool",
    "summary": "summary_tool",
}

async def route_query(query: str, classifier=initial_classifier):
    prediction = await classify_intent(query, classifier=classifier, run_name="router-classification")
    tool_name = INTENT_TO_TOOL[prediction.intent]
    tool = mcp_client.tool_by_name[tool_name]

    # The MCP tools have different input names.
    if prediction.intent == "summary":
        tool_input = {"text": query}
    else:
        tool_input = {"query": query}

    result = await tool.ainvoke(tool_input, config=lc_config(f"mcp-{tool_name}", ["mcp-tool", prediction.intent]))
    return {
        "query": query,
        "intent": prediction.intent,
        "confidence": prediction.confidence,
        "rationale": prediction.rationale,
        "tool_name": tool_name,
        "tool_result": result,
    }

examples = [
    "What is the purpose of MCP in tool-using AI agents?",
    " How much is 500 + 700.",
    "Summarize these notes: customer loved the demo but asked about security, SSO, and pricing.",
]

for q in examples:
    routed = asyncio.get_event_loop().run_until_complete(route_query(q))
    print("=" * 80)
    for k, v in routed.items():
        print(f"{k}: {v}")


query: What is the purpose of MCP in tool-using AI agents?
intent: info
confidence: 0.9
rationale: The user is asking for an explanation of a concept.
tool_name: info_tool
tool_result: [{'type': 'text', 'text': 'INFO TOOL RESULT\nReceived query: What is the purpose of MCP in tool-using AI agents?\nSuggested handling: provide facts, definitions, explanation, or guidance.', 'id': 'lc_f425515d-cd25-4ee9-99d3-ef7ddf2b587c'}]
query:  How much is 500 + 700.
intent: info
confidence: 0.8
rationale: The user is asking for a calculation result.
tool_name: info_tool
tool_result: [{'type': 'text', 'text': 'INFO TOOL RESULT\nReceived query:  How much is 500 + 700.\nSuggested handling: provide facts, definitions, explanation, or guidance.', 'id': 'lc_e6cd135a-3198-4c6b-bebe-b39b19cb4688'}]
query: Summarize these notes: customer loved the demo but asked about security, SSO, and pricing.
intent: summary
confidence: 0.8
rationale: The user asks to summarize the given notes.
tool_name: summary_tool
tool

# Step 8 a. Evaluate classifier accuracy

In [11]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

EVAL_DATA = [
    ("What is MCP and why is it useful for agents?", "info"),
    ("Explain the difference between LangChain and LangGraph.", "info"),
    ("How do I configure Langfuse tracing?", "info"),
    ("Compare Groq and Azure OpenAI for latency-sensitive workloads.", "info"),
    ("Create a Jira ticket for the failed login bug.", "action"),
    ("Send an email to the team about tomorrow's demo.", "action"),
    ("Book a meeting with the platform team next Monday.", "action"),
    ("Generate a CSV file from these records.", "action"),
    ("Summarize the following meeting transcript.", "summary"),
    ("Give me the key points from this support conversation.", "summary"),
    ("Condense this article into five bullets.", "summary"),
    ("Write an executive recap of this project update.", "summary"),
    ("Can you tell me what actions are available in this workflow?", "info"),
    ("Take these notes and produce a short summary.", "summary"),
    ("Update the customer record with the new phone number.", "action"),
]

async def evaluate_classifier(classifier, dataset=EVAL_DATA, run_prefix="eval"):
    rows = []
    for query, gold in dataset:
        pred = await classify_intent(query, classifier=classifier, run_name=f"{run_prefix}-{gold}")
        rows.append({
            "query": query,
            "gold": gold,
            "pred": pred.intent,
            "confidence": pred.confidence,
            "rationale": pred.rationale,
            "correct": pred.intent == gold,
        })
    df = pd.DataFrame(rows)
    acc = accuracy_score(df["gold"], df["pred"])
    print(f"Accuracy: {acc:.2%}")
    print("Classification report:")
    print(classification_report(df["gold"], df["pred"]))
    print("Confusion matrix:")
    print(confusion_matrix(df["gold"], df["pred"], labels=["info", "action", "summary"]))
    return df

initial_eval_df = asyncio.get_event_loop().run_until_complete(evaluate_classifier(initial_classifier, run_prefix="initial"))
initial_eval_df

Accuracy: 100.00%
Classification report:
              precision    recall  f1-score   support

      action       1.00      1.00      1.00         5
        info       1.00      1.00      1.00         5
     summary       1.00      1.00      1.00         5

    accuracy                           1.00        15
   macro avg       1.00      1.00      1.00        15
weighted avg       1.00      1.00      1.00        15

Confusion matrix:
[[5 0 0]
 [0 5 0]
 [0 0 5]]


,query,gold,pred,confidence,rationale,correct
0,What is MCP and why is it useful for agents?,info,info,0.8,The user is asking for an explanation of MCP a...,True
1,Explain the difference between LangChain and L...,info,info,0.9,The user is asking for an explanation of the d...,True
2,How do I configure Langfuse tracing?,info,info,0.8,The user is asking for guidance on configuring...,True
3,Compare Groq and Azure OpenAI for latency-sens...,info,info,0.9,The user is asking for a comparison between tw...,True
4,Create a Jira ticket for the failed login bug.,action,action,0.9,The user wants to create a Jira ticket.,True
5,Send an email to the team about tomorrow's demo.,action,action,0.9,"The user wants to send an email, which is an a...",True
6,Book a meeting with the platform team next Mon...,action,action,0.9,The user wants to schedule a meeting.,True
7,Generate a CSV file from these records.,action,action,0.8,The user wants a file to be generated.,True
8,Summarize the following meeting transcript.,summary,summary,0.9,The user asks for a summary of a meeting trans...,True
9,Give me the key points from this support conve...,summary,summary,0.8,The user wants key points extracted from a con...,True



# Step 9 - Improve Prompt (V2)

Create a better prompt with:

- clearer instructions
- intent definitions
- examples
- stricter output rules

In [12]:
improved_classifier_prompt = ChatPromptTemplate.from_messages([
    ("system", """
You are a strict intent classifier for an MCP routing agent.

Choose exactly one intent from: info, action, summary.

Definitions:
1. info
   The user wants knowledge, explanation, definitions, comparison, advice, diagnostics, or available options.
   Common cues: what, why, how, explain, compare, list, define, guide me, tell me.
   Important: asking what actions are possible is info, not action.

2. action
   The user wants the assistant or tool to perform an operation or change state.
   Common cues: create, send, update, delete, schedule, book, run, execute, generate a file, assign, open, close, upload.
   Use action even if the request contains some text to include in the action.

3. summary
   The user wants existing content compressed or transformed into a recap.
   Common cues: summarize, condense, recap, key points, TL;DR, executive summary, bullet summary.
   Use summary when the main deliverable is a shorter representation of provided content.

Tie-breakers:
- If the user asks to summarize and then send/update/create, classify as action because external execution is requested.
- If the user asks to explain how to do an action, classify as info because no execution is requested.
- If the user asks to create a summary document/file, classify as action because artifact creation is requested.

Return only the structured classification.
"""),
    ("human", "User query: {query}")
])

improved_classifier = improved_classifier_prompt | structured_llm

improved_eval_df = asyncio.get_event_loop().run_until_complete(evaluate_classifier(improved_classifier, run_prefix="improved"))
improved_eval_df


Accuracy: 93.33%
Classification report:
              precision    recall  f1-score   support

      action       0.83      1.00      0.91         5
        info       1.00      1.00      1.00         5
     summary       1.00      0.80      0.89         5

    accuracy                           0.93        15
   macro avg       0.94      0.93      0.93        15
weighted avg       0.94      0.93      0.93        15

Confusion matrix:
[[5 0 0]
 [0 5 0]
 [0 1 4]]


,query,gold,pred,confidence,rationale,correct
0,What is MCP and why is it useful for agents?,info,info,0.9,The user is asking for knowledge about MCP and...,True
1,Explain the difference between LangChain and L...,info,info,0.9,The user is asking for an explanation of the d...,True
2,How do I configure Langfuse tracing?,info,info,0.9,The user is asking for guidance on how to conf...,True
3,Compare Groq and Azure OpenAI for latency-sens...,info,info,0.9,The user is asking for a comparison between tw...,True
4,Create a Jira ticket for the failed login bug.,action,action,0.9,The user wants the assistant to perform an ope...,True
5,Send an email to the team about tomorrow's demo.,action,action,0.9,The user wants the assistant to perform an ope...,True
6,Book a meeting with the platform team next Mon...,action,action,0.9,"The user is requesting to book a meeting, whic...",True
7,Generate a CSV file from these records.,action,action,0.9,The user wants the assistant to create a CSV f...,True
8,Summarize the following meeting transcript.,summary,summary,1.0,The user explicitly asks to summarize a meetin...,True
9,Give me the key points from this support conve...,summary,summary,0.8,The user is asking for key points from a conve...,True


# Step 10. Compare initial(V1) vs improved prompt(V2)

In [13]:
comparison = initial_eval_df[["query", "gold", "pred", "correct"]].rename(columns={"pred": "initial_pred", "correct": "initial_correct"}).merge(
    improved_eval_df[["query", "pred", "correct"]].rename(columns={"pred": "improved_pred", "correct": "improved_correct"}),
    on="query"
)

initial_acc = comparison["initial_correct"].mean()
improved_acc = comparison["improved_correct"].mean()
print(f"Initial accuracy: {initial_acc:.2%}")
print(f"Improved accuracy: {improved_acc:.2%}")
comparison

Initial accuracy: 100.00%
Improved accuracy: 93.33%


,query,gold,initial_pred,initial_correct,improved_pred,improved_correct
0,What is MCP and why is it useful for agents?,info,info,True,info,True
1,Explain the difference between LangChain and L...,info,info,True,info,True
2,How do I configure Langfuse tracing?,info,info,True,info,True
3,Compare Groq and Azure OpenAI for latency-sens...,info,info,True,info,True
4,Create a Jira ticket for the failed login bug.,action,action,True,action,True
5,Send an email to the team about tomorrow's demo.,action,action,True,action,True
6,Book a meeting with the platform team next Mon...,action,action,True,action,True
7,Generate a CSV file from these records.,action,action,True,action,True
8,Summarize the following meeting transcript.,summary,summary,True,summary,True
9,Give me the key points from this support conve...,summary,summary,True,summary,True


# Step 11. Use imporved routing agent

In [14]:
test_queries = [
    "Explain how Langfuse helps debug an agent.",
    "Schedule a meeting with the data science team tomorrow at 3 PM.",
    "Summarize this: The release was delayed by two days because QA found a regression in login, but the fix is now merged.",
    "What are the steps to configure Azure OpenAI in LangChain?",
    "Create a short summary file from this transcript.",
]

for q in test_queries:
    routed = asyncio.get_event_loop().run_until_complete(route_query(q, classifier=improved_classifier))
    print("=" * 80)
    print("QUERY:", routed["query"])
    print("INTENT:", routed["intent"], "| CONFIDENCE:", routed["confidence"], "| TOOL:", routed["tool_name"])
    print("RATIONALE:", routed["rationale"])
    print(routed["tool_result"])

QUERY: Explain how Langfuse helps debug an agent.
INTENT: info | CONFIDENCE: 0.9 | TOOL: info_tool
RATIONALE: The user is asking for an explanation of how Langfuse helps debug an agent.
[{'type': 'text', 'text': 'INFO TOOL RESULT\nReceived query: Explain how Langfuse helps debug an agent.\nSuggested handling: provide facts, definitions, explanation, or guidance.', 'id': 'lc_74f2c512-5250-4dba-9c05-f33880288717'}]
QUERY: Schedule a meeting with the data science team tomorrow at 3 PM.
INTENT: action | CONFIDENCE: 0.99 | TOOL: action_tool
RATIONALE: The user is requesting to schedule a meeting, which is a specific action.
[{'type': 'text', 'text': 'ACTION TOOL RESULT\nReceived request: Schedule a meeting with the data science team tomorrow at 3 PM.\nSuggested handling: execute or prepare the requested operation after validation.', 'id': 'lc_b6e64632-0874-408c-aea6-274c95756481'}]
QUERY: Summarize this: The release was delayed by two days because QA found a regression in login, but the fix

# Step 12 - Observe Langfuse Traces

Go to Langfuse dashboard and inspect:

- prompts
- outputs
- latency
- token usage
- wrong predictions

Try analyzing:

- Why some classifications failed
- Whether prompt was too weak

In [15]:
if langfuse:
    langfuse.flush()
    print("Langfuse traces flushed.")
else:
    print("Langfuse is not configured, so there are no traces to flush.")

Langfuse traces flushed.
